In [7]:
!pip install streamlit

In [8]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [1]:
import streamlit as st
import mne
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import seaborn as sns
import pandas as pd

In [2]:

st.title("EEG Deep Learning App")

uploaded_file = st.file_uploader("Upload EDF File", type=["edf"])

2026-04-09 07:16:34.890 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.098 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-09 07:16:35.099 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.100 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.105 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.108 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.111 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-09 07:16:35.114 Thread 'MainThread': mi

In [3]:

if uploaded_file is not None:

    raw = mne.io.read_raw_edf(uploaded_file, preload=True)
    raw.pick_types(eeg=True)

    st.write("EEG Info:", raw.info)

    # Preprocessing
    raw.filter(0.5, 40)
    raw.notch_filter(20)
    raw.resample(100)

    # Epochs
    def create_epochs(raw, duration=2):
        events = mne.make_fixed_length_events(raw, duration=duration)
        epochs = mne.Epochs(raw, events, tmin=0, tmax=duration,
                            baseline=None, preload=True)
        return epochs

    epochs = create_epochs(raw)
    data = epochs.get_data()

    # Normalize
    data = (data - np.mean(data)) / np.std(data)

    st.write("Data Shape:", data.shape)

    # Plot EEG
    fig1, ax1 = plt.subplots()
    ax1.plot(data[0][0])
    ax1.set_title("EEG Signal")
    st.pyplot(fig1)

    # Dummy labels
    labels = np.random.randint(0, 2, len(data))

    # Model
    class EEG_CNN(nn.Module):
        def __init__(self, channels):
            super().__init__()
            self.conv1 = nn.Conv1d(channels, 16, 3)
            self.pool = nn.MaxPool1d(2)
            self.conv2 = nn.Conv1d(16, 32, 3)
            self.fc1 = nn.Linear(32 * 24, 64)
            self.fc2 = nn.Linear(64, 2)

        def forward(self, x):
            x = self.pool(torch.relu(self.conv1(x)))
            x = self.pool(torch.relu(self.conv2(x)))
            x = x.view(x.size(0), -1)
            x = torch.relu(self.fc1(x))
            return self.fc2(x)

    if st.button("Train Model"):

        channels = data.shape[1]
        model = EEG_CNN(channels)

        X = torch.tensor(data, dtype=torch.float32)
        y = torch.tensor(labels, dtype=torch.long)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        loss_list = []
        accuracy_list = []

        for epoch in range(5):
            outputs = model(X)
            loss = criterion(outputs, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs, axis=1)
            acc = (preds == y).float().mean().item()

            loss_list.append(loss.item())
            accuracy_list.append(acc)

            st.write(f"Epoch {epoch+1}, Loss: {loss.item()}")

        preds = preds.numpy()
        y_true = y.numpy()

        accuracy = (preds == y_true).mean()
        st.success(f"Accuracy: {accuracy:.2f}")

        # Confusion Matrix
        cm = confusion_matrix(y_true, preds)
        fig2, ax2 = plt.subplots()
        ConfusionMatrixDisplay(cm).plot(ax=ax2)
        st.pyplot(fig2)

        # ROC Curve
        probs = torch.softmax(outputs, dim=1)[:,1].detach().numpy()
        fpr, tpr, _ = roc_curve(y_true, probs)
        roc_auc = auc(fpr, tpr)

        fig3, ax3 = plt.subplots()
        ax3.plot(fpr, tpr)
        ax3.set_title(f"ROC Curve (AUC = {roc_auc:.2f})")
        st.pyplot(fig3)

        # Loss graph
        fig4, ax4 = plt.subplots()
        ax4.plot(loss_list)
        ax4.set_title("Training Loss")
        st.pyplot(fig4)

        # Accuracy graph
        fig5, ax5 = plt.subplots()
        ax5.plot(accuracy_list)
        ax5.set_title("Accuracy")
        st.pyplot(fig5)

        # Multi-channel EEG
        fig6, ax6 = plt.subplots()
        for i in range(min(5, data.shape[1])):
            ax6.plot(data[0][i] + i*5)
        ax6.set_title("Multi-Channel EEG")
        st.pyplot(fig6)

        # Heatmap
        sample_data = data[:200]
        df = pd.DataFrame({
            f"Ch_{i}": sample_data[:, i, :].mean(axis=1)
            for i in range(sample_data.shape[1])
        })
        df["Label"] = labels[:200]

        fig7, ax7 = plt.subplots()
        sns.heatmap(df.corr(), ax=ax7)
        ax7.set_title("Correlation Heatmap")
        st.pyplot(fig7)